# Notebook 09 — Human Review Simulation

Tasks:
- simulate review queue
- record decisions
- create feedback dataset
- agreement/error analysis

In [ ]:
import pandas as pd
import numpy as np
import json
from datetime import datetime

OUTPUT_DIR = "D:/data viewer/data/processed"

## Simulate Review Queue

In [ ]:
ai_output = pd.read_csv(f"{OUTPUT_DIR}/ai_reviewer_output.csv")
pairs = pd.read_csv(f"{OUTPUT_DIR}/entity_pairs.csv")

review_queue = ai_output.merge(pairs[['pair_id', 'label']], on='pair_id')
print(f"Review queue size: {len(review_queue)}")
print(f"\nClassification distribution:")
print(review_queue['classification'].value_counts())

## Record Decisions

In [ ]:
np.random.seed(42)
decisions = []

for _, row in review_queue.head(100).iterrows():
    if row['classification'] == 'ENTITY_MISMATCH':
        decision = 'REJECT / NON-MATCH'
    elif row['classification'] == 'NUMERIC_CONFLICT':
        decision = 'REQUEST CLARIFICATION'
    else:
        decision = 'APPROVE / MATCH'
    
    decisions.append({
        'pair_id': row['pair_id'],
        'reviewer_id': 'reviewer_001',
        'decision': decision,
        'reason': f"AI classification: {row['classification']}",
        'comment': '',
        'model_version': 'v1.0',
        'timestamp': datetime.now().isoformat(),
    })

decisions_df = pd.DataFrame(decisions)
print(f"Decisions recorded: {len(decisions_df)}")
print(decisions_df['decision'].value_counts())

## Agreement/Error Analysis

In [ ]:
decisions_df = decisions_df.merge(pairs[['pair_id', 'label']].rename(columns={'label': 'label_true'}), on='pair_id')

# Simulate agreement: approve = label 1, reject = label 0
decisions_df['agreed'] = (
    ((decisions_df['decision'] == 'APPROVE / MATCH') & (decisions_df['label_true'] == 1)) |
    ((decisions_df['decision'] == 'REJECT / NON-MATCH') & (decisions_df['label_true'] == 0))
)

agreement_rate = decisions_df['agreed'].mean()
print(f"Agreement rate: {agreement_rate:.2%}")
print(f"\nDisagreements: {(~decisions_df['agreed']).sum()}")
print(decisions_df[~decisions_df['agreed']][['pair_id', 'decision', 'label_true']].head(10))

## Save Feedback Dataset

In [ ]:
feedback = decisions_df[['pair_id', 'decision', 'reason', 'timestamp', 'model_version']].copy()
feedback.to_csv(f"{OUTPUT_DIR}/reviewer_feedback.csv", index=False)
print(f"Feedback dataset saved: {len(feedback)} rows")
print(feedback.head())